# Download Africa cultural POIs

Run from this project directory. The downloader reads `iso_3166_1_africa.json` and uses your five museum, library and archive tag filters. It has no third-party dependencies.

GeoJSON contains Point coordinates in **[longitude, latitude]** order. Nodes retain their coordinates; ways and relations use bounding-box centers, which can lie outside the actual shape. Original tags are in `properties.tags`. OSM coverage and tagging vary: this retrieves matching mapped features, not every real-world institution.

Requests run sequentially with pauses and retries. ISO-tagged administrative areas are selected without fixing admin_level to 2, so territories can match too. Missing boundaries are reported as failures rather than empty countries.

[Overpass output documentation](https://wiki.openstreetmap.org/wiki/Overpass_API/Overpass_QL). Data © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright), available under the ODbL.


In [ ]:
import importlib
import poi_download

# Reload changes when rerunning in an existing notebook kernel.
importlib.reload(poi_download)
from poi_download import download_africa_pois


Start with Lesotho to check connectivity and the resulting file. This writes to a separate directory from the full download.


In [ ]:
sample_report = download_africa_pois(country_codes=["LS"], output_dir="data/pois_sample")
sample_report


Run the following cell to download every entry in the supplied country list. Output: country files such as `LS.geojson`, a deduplicated `africa_pois.geojson`, and `download_report.json`. Check `complete` and `failed`: a combined file may contain only successful countries. The cell below resumes using existing country files and retries missing countries. Set `resume=False` to refresh them. Failed requests retry up to five times with exponential backoff, jitter, and progress messages. The download report is checkpointed after each country; rerun the cell after an interruption. All reused and newly downloaded countries are included in the combined file.


In [ ]:
report = download_africa_pois(
    resume=True,       # Reuse completed country files; retry missing countries.
    attempts=5,       # Total attempts per country.
    timeout=180,      # Server query timeout in seconds.
    retry_delay=30,   # Backoff: approximately 30, 60, 120, 240 seconds.
    max_retry_delay=300,
)
report


In [ ]:
import json
from pathlib import Path

pois = json.loads(Path("data/pois/africa_pois.geojson").read_text(encoding="utf-8"))
print(f"{len(pois['features'])} unique OSM locations")
print("Countries requiring review:", report["failed"])


## View the POIs with Folium
Run the install cell if Folium is missing. Generate and display the map directly in this notebook; the same map is saved to `data/africa_pois_map.html`. Category toggles and colored points show museums, libraries, and archives. Click a point for details. Internet access is needed for map assets and OpenStreetMap tiles.


In [9]:
%pip install "folium>=0.16,<1"


  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [folium]━━ 10/11 [folium]idna]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import importlib
import poi_map

importlib.reload(poi_map)
poi_map_view = poi_map.create_poi_map()
poi_map_view.save('data/africa_pois_map.html')
poi_map_view
